In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [2]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [35]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y


n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%

torch.Size([182778, 3]) torch.Size([182778])
torch.Size([22633, 3]) torch.Size([22633])
torch.Size([22735, 3]) torch.Size([22735])


In [39]:
#CLASSES

class Linear:

    def __init__(self, fan_in, fan_out, bias = True):
        self.weight = torch.randn(fan_in, fan_out)/ fan_out**0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, x):
        self.out = x @ self.weight

        if self.bias is not None:
            return self.out + self.bias
        return self.out

    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])

#----------------------------------------------------------------------------------------
        
class BatchNorm:

    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True
        # parameters
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
        # buffers
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.zeros(dim)


    def __call__(self, x):
        #calculate the forward pass
        if self.training:
            self.mean = x.mean(0, keepdim=True)
            self.var = x.var(0, keepdim=True)
        else:
            self.mean = self.running_mean
            self.var = self.running_var

        xhat = (x - self.mean) / torch.sqrt(self.var + self.eps)
        self.out = self.gamma * xhat + self.beta
        
        if self.training:
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * self.mean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * self.var
        
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]

#----------------------------------------------------------------------------------------


class Tanh:

    def __call__(self, x):
        self.out =  torch.tanh(x)
        return self.out

    def parameters(self):
        return []

#----------------------------------------------------------------------------------------

class Embedding:

    def __init__(self, num_embeddings, embedding_dim):
        self.weight = torch.randn(num_embeddings, embedding_dim)

    def __call__(self, IX):
        return self.weight[IX]

    def parameters(self):
        return [self.weight]
        
#----------------------------------------------------------------------------------------

class Flatten:
    def __call__(self, x):
        self.out = x.view(x.shape[0],-1)
        return self.out

    def parameters(self):
        return []
         

In [45]:
n_embd = 10  # dimension of a character embedding vector
n_hidden = 200 # the no. of neuron in hidden layer

layers = [
    Embedding(vocab_size, n_embd),
    Flatten(),
    Linear(n_embd * block_size, n_hidden, bias=False), BatchNorm(n_hidden), Tanh(),
    Linear(n_hidden, vocab_size, bias=False)
]

#parameter init
with torch.no_grad():
    layers[-1].weight *= 0.1

parameters = [p for layer in layers for p in layer.parameters()]
for p in parameters:
    p.requires_grad = True

In [46]:
# TRAINING

max_steps = 200000
batch_size = 32
lossi = []

for i in range(max_steps):

    #minibatch construction
    ix = torch.randint(0, Xtr.shape[0], (batch_size, ))
    Xb, Yb = Xtr[ix], Ytr[ix]

    #forward pass
    x = Xb
    for layer in layers:
        x = layer(x)
    loss = F.cross_entropy(x, Yb)
    
    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # weight updating
    lr = 0.1 if i < 150000 else 0.01 #step learning rate decay
    for p in parameters:
        p.data += -lr * p.grad

    #track Stats
    if i% 10000 == 0:
        print(f'{i} : loss = {loss.item() : .4f}')
        
    lossi.append(loss.log10().item())

0 : loss =  3.3339
10000 : loss =  2.3604
20000 : loss =  2.2537
30000 : loss =  2.2710
40000 : loss =  1.9362
50000 : loss =  1.8509
60000 : loss =  2.3506
70000 : loss =  2.3250
80000 : loss =  1.9239
90000 : loss =  2.3543
100000 : loss =  2.1624
110000 : loss =  1.8614
120000 : loss =  1.8259
130000 : loss =  2.4151
140000 : loss =  2.0385
150000 : loss =  1.8620
160000 : loss =  1.9161
170000 : loss =  2.2426
180000 : loss =  1.5417
190000 : loss =  1.7599
